In [5]:
import sqlite3
import pandas as pd

# Connect to SQLite database from data_generator
conn = sqlite3.connect('rideshare_telematics.db');
cursor = conn.cursor();

# Writing SQL queries directly in Python to bypass installation of SQL IDE

# Clean up database environment for running multiple times
cursor.execute("DROP TABLE IF EXISTS calculated_scores;");
cursor.execute("DROP TABLE IF EXISTS final_scores;");

# Data cleaning and calculating base risk score
sql_processing_query = """
CREATE TABLE IF NOT EXISTS calculated_scores AS
SELECT
    Driver_ID,
    Is_Full_Time,
    Annual_Miles,
    Speeding_Min_Per_100m,
    -- Default missing values to 0
    COALESCE(Hard_Brakes_Per_100m, 0) AS Brakes_Cleaned,
    COALESCE(Night_Driving_Pct, 0) AS Night_Pct_Cleaned,

    -- Actuarial Weights Formula:
    ((COALESCE(Hard_Brakes_Per_100m, 0) * 0.4) + 
    (Speeding_Min_Per_100m * 0.4) + 
    (COALESCE(Night_Driving_Pct, 0) * 20)) AS Base_Behavior_Score,

    -- Differentiate scores for different types of drivers
    CASE
        WHEN Is_Full_Time = 0 THEN 1.25   -- 25% risk surcharge for part-time drivers
        ELSE 0.90                         -- 10% loyalty discounty for full_time dirvers
    END AS Score_Modifier
FROM 
    raw_telematics;
""";
cursor.execute(sql_processing_query);

# Compute final risk score
sql_final_query = """
CREATE TABLE final_scores AS
SELECT *,
    (Base_Behavior_Score * Score_Modifier) AS Final_Risk_Score
FROM calculated_scores;
""";
cursor.execute(sql_final_query);

# Saves table into .db file
conn.commit();

# Extract SQL table to CSV for R
df_final = pd.read_sql_query("SELECT * FROM final_scores", conn);
df_final.to_csv('scored_drivers.csv', index=False);

# Close connection
conn.close()